# AutoML for EURUSD Forex Data

This notebook demonstrates how to perform automatic machine learning on EURUSD forex data using `auto-sklearn`. We will also incorporate external features and visualize the results.

In [ ]:
# Install necessary libraries
!pip install fastparquet pandas scikit-learn auto-sklearn ta matplotlib seaborn

## 1. Read Data

In [ ]:
import pandas as pd
import fastparquet as fp

# Read parquet file
file_path = '/opt/data/eurusd.parquet'
df = pd.read_parquet(file_path, engine='fastparquet')

# View the first few rows
df.head()

## 2. Data Preprocessing

In [ ]:
# Convert timestamp to datetime and set as index
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.set_index('timestamp', inplace=True)

# View the first few rows
df.head()

## 3. Read External Features

In [ ]:
import numpy as np

# Read libsvm file
def read_libsvm(filename):
    data = []
    with open(filename, 'r') as f:
        for line in f:
            parts = line.strip().split()
            label = int(parts[0])
            features = {}
            for part in parts[1:]:
                index, value = part.split(':')
                features[int(index)] = float(value)
            data.append((label, features))
    return data

# Read external features
feature_data = read_libsvm('feature.libsvm')

# Convert external features to DataFrame
external_features = pd.DataFrame([item[1] for item in feature_data], columns=range(14))

# Read metadata
import json
with open('feature.meta', 'r') as f:
    metadata = json.load(f)

# Map metadata column names to DataFrame
external_features.rename(columns={v: k for k, v in metadata.items()}, inplace=True)

# View the first few rows of external features
external_features.head()

## 4. Merge Features

In [ ]:
# Merge external features with existing dataset
df = df.join(external_features, how='inner')

# View the first few rows of merged dataset
df.head()

## 5. Feature Engineering

In [ ]:
import ta

# Define a function to calculate technical indicators
def calculate_indicators(df):
    df['MACD'] = ta.trend.macd_diff(df['close'])
    df['RSI'] = ta.momentum.rsi(df['close'])
    df['Bollinger_High'] = ta.volatility.bollinger_hband(df['close'])
    df['Bollinger_Low'] = ta.volatility.bollinger_lband(df['close'])
    df['DeMark'] = ta.trend.ichimoku_a(df['high'], df['low'])
    return df

# Define a function to extract features from different timeframes
def extract_features_from_timeframe(df, timeframe):
    df_resampled = df.resample(timeframe).agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    })
    df_resampled = calculate_indicators(df_resampled)
    return df_resampled

# Define timeframes
timeframes = ['3T', '5T', '15T', '30T', '1H', '2H', '4H', '1D']

# Extract features from different timeframes
features = {}
for tf in timeframes:
    features[tf] = extract_features_from_timeframe(df, tf)

# Merge features
merged_features = pd.concat(features.values(), axis=1, keys=features.keys())

# View the first few rows of merged features
merged_features.head()

## 6. Generate Target Variable

In [ ]:
# Calculate the next minute's price change
merged_features['target'] = (merged_features['3T']['close'].shift(-1) > merged_features['3T']['close']).astype(int)

# Drop the last row as it has no target variable
merged_features.dropna(inplace=True)

# View the first few rows
merged_features.head()

## 7. Split Data

In [ ]:
# Split data into training and testing sets
train_size = int(len(merged_features) * 0.8)
train_data = merged_features[:train_size]
test_data = merged_features[train_size:]

# Separate features and target variable
X_train = train_data.drop(columns=['target'])
y_train = train_data['target']

X_test = test_data.drop(columns=['target'])
y_test = test_data['target']

## 8. AutoML

In [ ]:
import autosklearn.classification

# Create an AutoML model
automl = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=120,  # Set time limit
    per_run_time_limit=30,        # Set time limit per model run
    tmp_folder='/tmp/autosklearn_classification_tmp',
    output_folder='/tmp/autosklearn_classification_out'
)

# Train the model
automl.fit(X_train, y_train)

# View the best models
print(automl.leaderboard())

## 9. Model Evaluation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Predict
y_pred = automl.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')

# Calculate confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)
print(f'Confusion Matrix:\n{conf_matrix}')

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=['Sell', 'Buy'], yticklabels=['Sell', 'Buy'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Print classification report
print(classification_report(y_test, y_pred, target_names=['Sell', 'Buy']))

# Feature importance analysis
from sklearn.inspection import permutation_importance

# Calculate feature importance
result = permutation_importance(automl, X_test, y_test, n_repeats=10, random_state=42)

# Get feature importance
feature_importances = pd.Series(result.importances_mean, index=X_test.columns)

# Plot feature importance
plt.figure(figsize=(10, 8))
feature_importances.sort_values(ascending=False).plot(kind='barh')
plt.title('Feature Importance')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

## 10. Result Analysis

In [ ]:
# Plot actual vs predicted values
plt.figure(figsize=(14, 7))
plt.plot(test_data.index, y_test, label='Actual')
plt.plot(test_data.index, y_pred, label='Predicted')
plt.legend()
plt.title('Actual vs Predicted')
plt.xlabel('Timestamp')
plt.ylabel('Target')
plt.show()

## 11. Save Model

In [ ]:
import joblib

# Save the model
joblib.dump(automl, 'eurusd_classification_model.pkl')

## 12. Load Model

In [ ]:
# Load the model
loaded_model = joblib.load('eurusd_classification_model.pkl')

# Use the loaded model for prediction
y_pred_loaded = loaded_model.predict(X_test)

## Summary

This notebook demonstrates how to perform automatic machine learning on EURUSD forex data using `auto-sklearn`. We have incorporated external features, visualized the results, and saved the model for future use.